# Instalação de pacotes

In [ ]:
!pip install openai scikit-learn pandas Groq kaggle

# Imports

In [ ]:
from google.colab import userdata

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = userdata.get('API_Key_GROQ')

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('API_Kaggle')

In [ ]:
import pandas as pd
import numpy as np

# Download da base

In [ ]:
!kaggle competitions download -c home-credit-default-risk

In [ ]:
!unzip home-credit-default-risk.zip

In [ ]:
df = pd.read_csv("application_train.csv")
df.head()

# Selecionar variáveis

In [ ]:
cols = [
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

df = df[cols].copy()

# Na base Home Credit, 365243 representa um valor especial para clientes sem emprego registrado.
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

df.head()

# Feature Engineering

* DTI → quanto da renda está comprometida
* credit_income_ratio → tamanho do crédito vs renda

In [ ]:
df["dti"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["credit_income_ratio"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

# Train

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocess_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

X_train_lr = preprocess_lr.fit_transform(X_train)
X_test_lr = preprocess_lr.transform(X_test)

X_train_tree = preprocess_tree.fit_transform(X_train)
X_test_tree = preprocess_tree.transform(X_test)

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

modelLR = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelLR.fit(X_train_lr, y_train)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

modelRF = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
modelRF.fit(X_train_tree, y_train)

# Avaliação dos modelos

Como a base é desbalanceada, a AUC é importante, mas não basta sozinha. Também vamos olhar precisão, recall e matriz de confusão para entender melhor o comportamento do modelo na classe de inadimplência.

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
import matplotlib.pyplot as plt

predsLR = modelLR.predict_proba(X_test_lr)[:, 1]
predsRF = modelRF.predict_proba(X_test_tree)[:, 1]

print(f"Taxa de inadimplência no teste: {y_test.mean():.2%}")
print('Regressão Logística - AUC: ', roc_auc_score(y_test, predsLR))
print('Random Forest - AUC: ', roc_auc_score(y_test, predsRF))

modelo_escolhido = modelLR
preds_modelo = predsLR
y_pred = (preds_modelo >= 0.5).astype(int)

print("\nRelatório de classificação - Regressão Logística")
print(classification_report(y_test, y_pred, digits=3))

print("Matriz de confusão - Regressão Logística")
print(confusion_matrix(y_test, y_pred))

RocCurveDisplay.from_predictions(y_test, predsLR, name="Regressão Logística")
RocCurveDisplay.from_predictions(y_test, predsRF, name="Random Forest", ax=plt.gca())
plt.title("Curva ROC")
plt.show()

# Explicabilidade dos modelos

Nesta etapa, vamos olhar quais variáveis mais pesam nos modelos. Isso ajuda a transformar a previsão em uma análise de crédito mais interpretável.

In [ ]:
coeficientes_lr = pd.DataFrame({
    "feature": X.columns,
    "coeficiente_regressao_logistica": modelLR.coef_[0]
})
coeficientes_lr["impacto_absoluto"] = coeficientes_lr["coeficiente_regressao_logistica"].abs()
coeficientes_lr = coeficientes_lr.sort_values("impacto_absoluto", ascending=False)

importancias_rf = pd.DataFrame({
    "feature": X.columns,
    "importancia_random_forest": modelRF.feature_importances_
}).sort_values("importancia_random_forest", ascending=False)

principais_fatores_modelo = coeficientes_lr.head(5)["feature"].tolist()

display(coeficientes_lr)
display(importancias_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coeficientes_lr.sort_values("coeficiente_regressao_logistica").plot.barh(
    x="feature",
    y="coeficiente_regressao_logistica",
    ax=axes[0],
    legend=False,
    title="Coeficientes - Regressão Logística"
)

importancias_rf.sort_values("importancia_random_forest").plot.barh(
    x="feature",
    y="importancia_random_forest",
    ax=axes[1],
    legend=False,
    title="Importância - Random Forest"
)

plt.tight_layout()
plt.show()

# Teste Cliente Real

In [ ]:
def classificar_risco(prob):
    if prob < 0.10:
        return "Baixo risco"
    if prob < 0.20:
        return "Risco moderado"
    return "Alto risco"

cliente = X_test.iloc[[0]]
cliente_lr = preprocess_lr.transform(cliente)
prob_default = modelLR.predict_proba(cliente_lr)[0][1]
risco = classificar_risco(prob_default)

print(f"Probabilidade de inadimplência: {prob_default:.2%}")
print(f"Classificação: {risco}")
cliente

In [ ]:
def criar_prompt(cliente, prob, risco):
    dados_cliente = cliente.to_dict(orient="records")[0]
    return f"""
Você é um analista de crédito de um grande banco.

Dados do cliente:
{dados_cliente}

Probabilidade estimada de inadimplência: {prob:.2%}
Classificação preliminar do modelo: {risco}
Variáveis mais relevantes para o modelo: {principais_fatores_modelo}

Considere boas práticas de crédito, mas trate o modelo como apoio à decisão, não como decisão automática.

Responda:
- Classificação de risco
- Principais fatores observados nos dados
- Recomendação
- Justificativa profissional
- Cuidados ou informações adicionais que o banco deveria verificar
"""

In [ ]:
import groq
import os

client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])

prompt = criar_prompt(cliente, prob_default, risco)

response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "Você é um especialista em risco de crédito."},
        {"role": "user", "content": prompt}
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

analise = response.choices[0].message.content
print(analise)